# 04_reference_data_load

## Purpose

Load reference CSV files from the governed landing volume into the Bronze reference Delta table.

Reference data is usually smaller and simpler than event streams, but it is still important because it supports later joins, validation, and enrichment.

In [0]:
from pyspark.sql import functions as F

In [0]:
# Project configuration

catalog = "vattenfall_dev"
schema = "raw"

landing_volume = "landing"

source_domain = "reference"

landing_subpath = "reference"

bronze_table_name = "bronze_asset_reference"
bronze_table = f"{catalog}.{schema}.{bronze_table_name}"

In [0]:
# Build governed Unity Catalog volume path

landing_path = f"/Volumes/{catalog}/{schema}/{landing_volume}/{landing_subpath}"

print("Source domain:", source_domain)
print("Landing path:", landing_path)
print("Bronze target table:", bronze_table)

In [0]:
# Inspect reference landing files before loading

display(dbutils.fs.ls(landing_path))

In [0]:
# Load reference CSV files with a simple Bronze Delta load pattern

reference_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(landing_path)
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

In [0]:
# Reset Day 1 prototype table before creating the Day 2 Bronze reference table

spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")

print("Dropped existing Bronze table if present:", bronze_table)

In [0]:
# Write reference records to Bronze Delta table

(
    reference_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print("Bronze reference load completed for:", source_domain)
print("Bronze target table:", bronze_table)

In [0]:
# Validate Bronze reference table

bronze_df = spark.table(bronze_table)

print("Rows in bronze after load:", bronze_df.count())
print("Columns in bronze:", bronze_df.columns)

display(bronze_df.limit(20))

In [0]:
# Check ingestion metadata columns

required_metadata_columns = ["ingestion_ts", "source_file"]

for column_name in required_metadata_columns:
    if column_name in bronze_df.columns:
        print(f"Metadata column present: {column_name}")
    else:
        raise ValueError(f"Missing metadata column: {column_name}")